**Table des matières**

- [1) Objectif du notebook et plan de formation](#objectif)
- [2) Rappel RAG et rôle de LlamaIndex](#rappel-rag)
- [3) Pré-requis et installation](#prerequis)
- [4) Vérification des données du projet](#verification-donnees)
- [5) Démo RAG minimale avec LlamaIndex (OpenAI)](#rag-openai)
- [6) Personnalisation du pipeline (chunking + retrieval)](#personnalisation)
- [7) Démo RAG locale avec Ollama](#rag-ollama)
- [8) Embeddings locaux avec OllamaEmbedding](#embeddings-ollama)
- [9) Contrôle qualité et bonnes pratiques](#qualite)
- [10) Conclusion](#conclusion)


<a id='objectif'></a>
# 1) Objectif du notebook et plan de formation

Ce notebook a pour objectif de démontrer, de manière **pratique et professionnelle**, la construction d’un assistant RAG simple avec **LlamaIndex** sur un corpus documentaire robotique.

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

1. Expliquer le workflow RAG (Retrieval, Augmentation, Generation).
2. Construire un pipeline RAG minimal avec LlamaIndex.
3. Configurer une variante cloud (OpenAI) et une variante locale (Ollama).
4. Ajuster les paramètres clés (chunking, top-k, mode de réponse).
5. Évaluer la qualité des réponses et contrôler les sources utilisées.

## Plan de formation recommandé

- **Module A (fondations)** : concepts RAG + composants LlamaIndex.
- **Module B (mise en pratique)** : indexation, questions/réponses, inspection des sources.
- **Module C (industrialisation)** : configuration locale, optimisation, gouvernance LLMOps.


<a id='rappel-rag'></a>
# 2) Rappel RAG et rôle de LlamaIndex

Le RAG combine :

1. **Retrieval** : récupération des passages pertinents dans une base documentaire.
2. **Augmentation** : injection de ces passages dans le prompt.
3. **Generation** : génération de réponse contextualisée par le LLM.

![Schéma simple d’une architecture RAG](./images/simple-rag.svg)

![Avant / après RAG](./images/avant-apres-rag.png)

## Pourquoi LlamaIndex ?

LlamaIndex fournit les briques nécessaires pour connecter des données métier à un LLM :

| Composant | Rôle | Exemples |
|---|---|---|
| Reader | Chargement des données | `SimpleDirectoryReader` |
| Node Parser | Découpage en chunks | `SentenceSplitter` |
| Embedding Model | Vectorisation | `OpenAIEmbedding`, `OllamaEmbedding` |
| Index | Structuration et recherche | `VectorStoreIndex` |
| Query Engine | Exécution des requêtes | `index.as_query_engine()` |


<a id='prerequis'></a>
# 3) Pré-requis et installation

## Pré-requis techniques

- Python 3.10+ (idéalement environnement virtuel `.venv`).
- Dossier `./data` contenant les documents métier (PDF, images, etc.).
- Pour OpenAI : variable d’environnement `OPENAI_API_KEY`.
- Pour Ollama : service local démarré (`http://localhost:11434`) et modèles téléchargés.


In [1]:
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Notebook path exists: {Path('.').resolve().exists()}")


Python: 3.14.6
Notebook path exists: True


In [1]:
# Installation des dépendances principales (à exécuter une seule fois par environnement)
%pip install -U llama-index llama-index-llms-openai llama-index-embeddings-openai
%pip install -U llama-index-llms-ollama llama-index-embeddings-ollama ollama
%pip install -U python-dotenv tqdm


  Using cached llama_index_llms_openai-0.8.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached llama_index_embeddings_openai-0.7.0-py3-none-any.whl.metadata (401 bytes)
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


<a id='verification-donnees'></a>
# 4) Vérification des données du projet

Cette cellule valide l’existence du corpus avant l’indexation.


In [3]:
from pathlib import Path

DATA_DIR = Path("./data")
assert DATA_DIR.exists(), f"Dossier introuvable: {DATA_DIR.resolve()}"

files = sorted([p for p in DATA_DIR.glob("**/*") if p.is_file()])
print(f"Nombre total de fichiers: {len(files)}")

by_ext = {}
for f in files:
    ext = f.suffix.lower() if f.suffix else "[sans extension]"
    by_ext[ext] = by_ext.get(ext, 0) + 1

print("\nRépartition par extension:")
for ext, count in sorted(by_ext.items()):
    print(f"- {ext}: {count}")

print("\nExemples de fichiers:")
for f in files[:10]:
    print("-", f)


Nombre total de fichiers: 19

Répartition par extension:
- .jpeg: 2
- .jpg: 1
- .pdf: 13
- .png: 1
- .webp: 2

Exemples de fichiers:
- data\robotique_doc_00.pdf
- data\robotique_doc_01.pdf
- data\robotique_doc_02.pdf
- data\robotique_doc_03.pdf
- data\robotique_doc_04.pdf
- data\robotique_doc_05.pdf
- data\robotique_doc_06.pdf
- data\robotique_doc_07.pdf
- data\robotique_doc_08.pdf
- data\robotique_doc_10.pdf


<a id='rag-openai'></a>
# 5) Démo RAG minimale avec LlamaIndex (OpenAI)

Chargement des variables d'environnements:

In [4]:
# Chargement des variables d'environnements:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from a .env file and sets them in os.environ

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "OPENAI_API_KEY is not set in the environment"

OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")
assert OLLAMA_API_KEY is not None, "OLLAMA_API_KEY is not set in the environment"

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
assert OLLAMA_BASE_URL is not None, "OLLAMA_BASE_URL is not set in the environment"



Cette section montre la version la plus concise d’un RAG LlamaIndex.


In [5]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY n'est pas défini. Configurez la variable d'environnement avant d'exécuter cette section.")

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1, api_key=OPENAI_API_KEY, timeout=60)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

documents = SimpleDirectoryReader(str(DATA_DIR)).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4, response_mode="compact")

print(f"Documents indexés: {len(documents)}")


Retrying llama_index.embeddings.openai.base.OpenAIEmbedding._get_text_embeddings.<locals>._retryable_get_embeddings in 1 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}.


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
questions = [
    "Quels sont les principes de base en robotique ?",
    "Quels sont les défis actuels en robotique ?",
    "Quels sont les derniers développements en robotique ?",
]

for i, q in enumerate(questions, start=1):
    r = query_engine.query(q)
    print(f"
--- Question {i} ---")
    print(q)
    print("Réponse:")
    print(str(r))


In [ ]:
# Inspection des passages sources pour vérifiabilité
sample_question = "Comment la cinématique est-elle présentée dans les documents ?"
sample_response = query_engine.query(sample_question)

print("Réponse:")
print(sample_response)

print("
Top passages sources:")
for rank, node in enumerate(sample_response.source_nodes[:3], start=1):
    print(f"
[{rank}] score={node.score:.4f}")
    excerpt = node.node.get_text().strip().replace("
", " ")
    print(excerpt[:350] + ("..." if len(excerpt) > 350 else ""))


<a id='personnalisation'></a>
# 6) Personnalisation du pipeline (chunking + retrieval)

Le code suivant illustre une configuration plus fine (chunking explicite et paramètres de récupération).


In [13]:
from llama_index.core.node_parser import SentenceSplitter

node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = node_parser.get_nodes_from_documents(documents, show_progress=True)

custom_index = VectorStoreIndex(nodes, show_progress=True)
custom_query_engine = custom_index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact",
)

test_query = "Quels éléments de base faut-il maîtriser en robotique ?"
custom_response = custom_query_engine.query(test_query)
print(custom_response)


e:\Formations\Jedha\03_Lead\M05L_LLMOps\LlamaIndex\demos\basic_RAG_with_LlamaIndex\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: OPENAI_A**_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

<a id='rag-ollama'></a>
# 7) Démo RAG locale avec Ollama

Cette variante est utile pour des environnements privés (sans envoi de données vers une API externe).

## Préparation locale

1. Installer Ollama : <https://github.com/ollama/ollama>
2. Démarrer le service Ollama (port par défaut `11434`).
3. Télécharger les modèles souhaités.


In [ ]:
# Exemples de modèles (à adapter selon votre machine)
# !ollama pull llama3.2:latest
# !ollama pull nomic-embed-text:latest


In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

OLLAMA_GENERATION_MODEL = "llama3.2:latest"
OLLAMA_EMBED_MODEL = "nomic-embed-text:latest"

Settings.llm = Ollama(
    model=OLLAMA_GENERATION_MODEL,
    request_timeout=180.0,
    context_window=8192,
)
Settings.embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

ollama_docs = SimpleDirectoryReader(str(DATA_DIR)).load_data()
ollama_index = VectorStoreIndex.from_documents(ollama_docs)
ollama_query_engine = ollama_index.as_query_engine(similarity_top_k=4, response_mode="compact")

print(f"Index local prêt avec {len(ollama_docs)} documents.")


In [ ]:
ollama_question = "Quelles applications concrètes de la robotique sont mentionnées ?"
ollama_response = ollama_query_engine.query(ollama_question)
print(ollama_response)


<a id='embeddings-ollama'></a>
# 8) Embeddings locaux avec OllamaEmbedding

Cette section vérifie la génération de vecteurs d’embedding en local.


In [ ]:
embedding_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

batch_embeddings = embedding_model.get_text_embedding_batch(
    [
        "La robotique mobile combine perception, planification et contrôle.",
        "Les capteurs sont essentiels pour la navigation autonome.",
    ],
    show_progress=True,
)

print(f"Dimension du vecteur: {len(batch_embeddings[0])}")
print("10 premières valeurs:", batch_embeddings[0][:10])


<a id='qualite'></a>
# 9) Contrôle qualité et bonnes pratiques

## Checklist de validation

- Le dossier `./data` est détecté et contient des documents exploitables.
- L’index se construit sans erreur.
- Les réponses sont cohérentes avec le corpus.
- Les `source_nodes` confirment la traçabilité.
- Les paramètres (`chunk_size`, `top_k`, modèles) sont explicitement documentés.

## Erreurs fréquentes

- `OPENAI_API_KEY` non défini.
- Service Ollama non démarré (`Connection refused` sur `localhost:11434`).
- Modèle Ollama non téléchargé.
- Corpus vide ou format documentaire non supporté.


<a id='conclusion'></a>
# 10) Conclusion

Ce notebook démontre un workflow RAG complet et reproductible avec LlamaIndex :

- une version **cloud** (OpenAI),
- une version **locale** (Ollama),
- une logique de validation orientée qualité (sources, paramètres, reproductibilité).

Il constitue une base solide pour :

1. un usage portfolio,
2. des ateliers de formation professionnelle,
3. une montée en complexité vers des architectures RAG de production (vector DB, évaluation automatique, observabilité, gouvernance LLMOps).
